In [1]:
import pandas as pd
import numpy as np
import os ## for file paths
from sqlalchemy import create_engine 

In [2]:
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
RAW_DIR = os.path.join(BASE_DIR, '..', 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, '..', 'data', 'processed')
DB_PATH = os.path.join(BASE_DIR, '..', 'data', 'bluestock_mf.db')

print("Paths set successfully")

Paths set successfully


In [3]:
nav=pd.read_csv(os.path.join(RAW_DIR,'02_nav_history.csv'))
print("Shape before cleaning:",nav.shape)
print("\n DataTypes ",nav.dtypes)

Shape before cleaning: (46000, 3)

 DataTypes  amfi_code      int64
date          object
nav          float64
dtype: object


There are 46000 rows and 3 columns

In [4]:
nav['date']=pd.to_datetime(nav['date'])
print("Date datatype after conversion :",nav['date'].dtype)
print("Date range :",nav['date'].min(),"to",nav['date'].max())

Date datatype after conversion : datetime64[ns]
Date range : 2022-01-03 00:00:00 to 2026-05-29 00:00:00


Converted date to proper datetime format

In [5]:
nav=nav.sort_values(['amfi_code','date']).reset_index(drop=True)
print("Sorted SUccessfully")
print(nav.head())

Sorted SUccessfully
   amfi_code       date       nav
0     100016 2022-01-03  520.4608
1     100016 2022-01-04  515.0971
2     100016 2022-01-05  521.7239
3     100016 2022-01-06  515.7880
4     100016 2022-01-07  515.1639


In [6]:
# Create full date range for each fund
nav = nav.set_index('date')
nav = nav.groupby('amfi_code').apply(lambda x: x.reindex(
    pd.date_range(x.index.min(), x.index.max(), freq='D')
)).drop(columns='amfi_code').reset_index()
nav.columns = ['amfi_code', 'date', 'nav']
nav['nav'] = nav.groupby('amfi_code')['nav'].ffill()
print("Shape after forward-fill:", nav.shape)
print("Null NAV values remaining:", nav['nav'].isnull().sum())

Shape after forward-fill: (64320, 3)
Null NAV values remaining: 0


C:\Users\Mahima Kulkarni\AppData\Local\Temp\ipykernel_8108\1288492888.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nav = nav.groupby('amfi_code').apply(lambda x: x.reindex(


In [7]:
transactions=pd.read_csv(os.path.join(os.path.join(RAW_DIR,'08_investor_transactions.csv')))
print("Shape before cleaning :",transactions.shape)
print("\n Data types :\n",transactions.dtypes)

Shape before cleaning : (32778, 13)

 Data types :
 investor_id            object
transaction_date       object
amfi_code               int64
transaction_type       object
amount_inr              int64
state                  object
city                   object
city_tier              object
age_group              object
gender                 object
annual_income_lakh    float64
payment_mode           object
kyc_status             object
dtype: object


In [8]:
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
print("Date dtype:", transactions['transaction_date'].dtype)
print("\nUnique transaction types:", transactions['transaction_type'].unique())
print("\nUnique KYC status:", transactions['kyc_status'].unique())

Date dtype: datetime64[ns]

Unique transaction types: ['SIP' 'Redemption' 'Lumpsum']

Unique KYC status: ['Verified' 'Pending']


Task 1 — Clean nav_history.csv Step 1: Date Conversion Converted date column from text (object) to datetime64 Date range intact: 2022-01-03 to 2026-05-29 Step 2: Sort Sorted by amfi_code and date (oldest to newest within each fund) Step 3: Forward-fill Missing NAV Original shape: (46000, 3) Shape after forward-fill: (64320, 3) Added 18,320 rows for weekends and holidays Forward-filled using previous trading day's NAV 0 null NAV values remaining

In [9]:
invalid_amount=transactions[transactions['amount_inr']<=0]
print("invalid amounts (<=0):",len(invalid_amount))
before= transactions.shape[0]
transactions=transactions.drop_duplicates()
after=transactions.shape[0]
print("Duplicates removed :",before-after)
print("\n Final shape:",transactions.shape)

invalid amounts (<=0): 0
Duplicates removed : 0

 Final shape: (32778, 13)


In [10]:
transactions.to_csv(os.path.join(PROCESSED_DIR, 'clean_transactions.csv'), index=False)
print("Saved clean_transactions.csv successfully")

Saved clean_transactions.csv successfully


In [11]:
# Remove duplicates
before = nav.shape[0]
nav = nav.drop_duplicates(subset=['amfi_code', 'date'])
after = nav.shape[0]
print(f"Duplicates removed: {before - after}")

# Validate NAV > 0
invalid_nav = nav[nav['nav'] <= 0]
print(f"Invalid NAV values (<=0): {len(invalid_nav)}")

print(f"\nFinal shape: {nav.shape}")

Duplicates removed: 0
Invalid NAV values (<=0): 0

Final shape: (64320, 3)


In [12]:
# Save cleaned nav_history to processed folder
nav.to_csv(os.path.join(PROCESSED_DIR, 'clean_nav.csv'), index=False)
print("Saved clean_nav.csv successfully")
print("Shape:", nav.shape)

Saved clean_nav.csv successfully
Shape: (64320, 3)


Step 4: Remove Duplicates and Validate NAV
0 duplicates found and removed
0 invalid NAV values (all NAV > 0)
Final clean shape: (64320, 3)
Saved as clean_nav.csv in data/processed/

In [13]:
transactions.to_csv(os.path.join(PROCESSED_DIR, 'clean_transactions.csv'), index=False)
print("Saved clean_transactions.csv successfully")

Saved clean_transactions.csv successfully


TASK 2
Clean investor_transactions.csv

Shape: 32778 rows, 13 columns
Date converted from text to datetime64
Transaction types already standardised: SIP, Lumpsum, Redemption
KYC status values valid: Verified, Pending only
0 invalid amounts
0 duplicates
Saved as clean_transactions.csv in data/processed/

In [14]:
perf=pd.read_csv(os.path.join(RAW_DIR,'07_scheme_performance.csv'))
print("Shape before cleaning :",perf.shape)
print("\n DataTypes",perf.dtypes)

Shape before cleaning : (40, 19)

 DataTypes amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object


In [15]:
# Check for nulls in return columns
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 
               'sharpe_ratio', 'alpha', 'beta', 'max_drawdown_pct']
print("Nulls in return columns:\n", perf[return_cols].isnull().sum())

# Check expense ratio range
print("\nExpense ratio range:")
print("Min:", perf['expense_ratio_pct'].min())
print("Max:", perf['expense_ratio_pct'].max())

# Flag out of range expense ratios
invalid_expense = perf[(perf['expense_ratio_pct'] < 0.1) | (perf['expense_ratio_pct'] > 2.5)]
print("\nFunds with invalid expense ratio:", len(invalid_expense))

# Flag negative Sharpe ratios
negative_sharpe = perf[perf['sharpe_ratio'] < 0]
print("Funds with negative Sharpe ratio:", len(negative_sharpe))

Nulls in return columns:
 return_1yr_pct      0
return_3yr_pct      0
return_5yr_pct      0
sharpe_ratio        0
alpha               0
beta                0
max_drawdown_pct    0
dtype: int64

Expense ratio range:
Min: 0.55
Max: 1.64

Funds with invalid expense ratio: 0
Funds with negative Sharpe ratio: 0


In [16]:
perf.to_csv(os.path.join(PROCESSED_DIR, 'clean_performance.csv'), index=False)
print("Saved clean_performance.csv successfully")

Saved clean_performance.csv successfully


Task 3 — Clean scheme_performance.csv

Shape: 40 rows, 19 columns
0 nulls in all return and risk columns
Expense ratio range: 0.55% to 1.64% — within valid SEBI range (0.1% to 2.5%)
0 funds with invalid expense ratio
0 funds with negative Sharpe ratio
No cleaning required — data is already valid
Saved as clean_performance.csv in data/processed/

In [ ]:
schema_sql = """
-- dim_fund
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code TEXT PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    benchmark TEXT,
    expense_ratio_pct REAL,
    risk_category TEXT,
    fund_manager TEXT
);

-- dim_date
CREATE TABLE IF NOT EXISTS dim_date (
    date_id TEXT PRIMARY KEY,
    year INTEGER,
    month INTEGER,
    quarter INTEGER,
    is_weekday INTEGER
);

-- fact_nav
CREATE TABLE IF NOT EXISTS fact_nav (
    amfi_code TEXT,
    date TEXT,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- fact_transactions
CREATE TABLE IF NOT EXISTS fact_transactions (
    investor_id TEXT,
    transaction_date TEXT,
    amfi_code TEXT,
    transaction_type TEXT,
    amount_inr INTEGER,
    state TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- fact_performance
CREATE TABLE IF NOT EXISTS fact_performance (
    amfi_code TEXT,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    sharpe_ratio REAL,
    alpha REAL,
    beta REAL,
    max_drawdown_pct REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- fact_aum
CREATE TABLE IF NOT EXISTS fact_aum (
    fund_house TEXT,
    date TEXT,
    aum_lakh_crore REAL,
    aum_crore INTEGER,
    num_schemes INTEGER
);
"""

# Save schema to sql folder
sql_path = os.path.join(BASE_DIR, '..', 'sql', 'schema.sql')
with open(sql_path, 'w') as f:
    f.write(schema_sql)
print("schema.sql saved successfully")

Task 4 — SQLite Star Schema Design

Designed 6 table star schema
2 dimension tables: dim_fund, dim_date
4 fact tables: fact_nav, fact_transactions, fact_performance, fact_aum
Primary keys defined on all dimension tables
Foreign keys link all fact tables to dim_fund via amfi_code
Schema saved as schema.sql in sql/ folder

In [19]:
# Create database connection
engine = create_engine(f'sqlite:///{DB_PATH}')
print("Database created at:", DB_PATH)

Database created at: C:\Users\Mahima Kulkarni\bluestock_mf_capstone\notebooks\..\data\bluestock_mf.db


In [20]:
fund_master=pd.read_csv(os.path.join(RAW_DIR,'01_fund_master.csv'))
fund_master['amfi_code']=fund_master['amfi_code'].astype(str)
fund_master.to_sql('dim_fund',engine,if_exists='replace',index=False)
print("dim_fund loaded :",len(fund_master),"rows")

dim_fund loaded : 40 rows


In [21]:
# Load clean nav as fact_nav
nav = pd.read_csv(os.path.join(PROCESSED_DIR, 'clean_nav.csv'))
nav['amfi_code'] = nav['amfi_code'].astype(str)
nav.to_sql('fact_nav', engine, if_exists='replace', index=False)
print("fact_nav loaded:", len(nav), "rows")

fact_nav loaded: 64320 rows


In [22]:
# Load clean transactions as fact_transactions
transactions = pd.read_csv(os.path.join(PROCESSED_DIR, 'clean_transactions.csv'))
transactions['amfi_code'] = transactions['amfi_code'].astype(str)
transactions.to_sql('fact_transactions', engine, if_exists='replace', index=False)
print("fact_transactions loaded:", len(transactions), "rows")

fact_transactions loaded: 32778 rows


In [23]:
# Load clean performance as fact_performance
perf = pd.read_csv(os.path.join(PROCESSED_DIR, 'clean_performance.csv'))
perf['amfi_code'] = perf['amfi_code'].astype(str)
perf.to_sql('fact_performance', engine, if_exists='replace', index=False)
print("fact_performance loaded:", len(perf), "rows")

fact_performance loaded: 40 rows


In [25]:
# Load aum as fact_aum
aum = pd.read_csv(os.path.join(RAW_DIR, '03_aum_by_fund_house.csv'))
aum.to_sql('fact_aum', engine, if_exists='replace', index=False)
print("fact_aum loaded:", len(aum), "rows")

fact_aum loaded: 90 rows


In [26]:
import sqlite3

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

tables = ['dim_fund', 'fact_nav', 'fact_transactions', 'fact_performance', 'fact_aum']
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table}: {count} rows")

conn.close()

dim_fund: 40 rows
fact_nav: 64320 rows
fact_transactions: 32778 rows
fact_performance: 40 rows
fact_aum: 90 rows


## Task 5 — Load Data into SQLite
- Created bluestock_mf.db in data/ folder
- Loaded 5 tables into database
- dim_fund: 40 rows
- fact_nav: 64,320 rows
- fact_transactions: 32,778 rows
- fact_performance: 40 rows
- fact_aum: 90 rows
- All row counts verified and match source CSVs

TASK 6

Query 1 Top 5 funds by AUM
Shows which funds are the largest. Investors use this to identify popular, well-managed funds.


In [29]:
import sqlite3

conn=sqlite3.connect(DB_PATH)
query1="""
SELECT scheme_name,fund_house,aum_crore
FROM fact_performance
ORDER BY aum_crore DESC
LIMIT 5;
"""
result1=pd.read_sql(query1,conn)
print(result1)

                                         scheme_name         fund_house  \
0  Mirae Asset Emerging Bluechip Fund - Regular -...     Mirae Asset MF   
1      Kotak Emerging Equity Fund - Regular - Growth  Kotak Mahindra MF   
2     Nippon India Small Cap Fund - Regular - Growth    Nippon India MF   
3         DSP Top 100 Equity Fund - Regular - Growth    DSP Mutual Fund   
4                UTI Mid Cap Fund - Regular - Growth    UTI Mutual Fund   

   aum_crore  
0      49046  
1      47469  
2      43630  
3      41828  
4      41728  


In [33]:
query2="""
SELECT substr(date,1,7) as month,
        ROUND(AVG(nav),2) as avg_nav
FROM fact_nav
GROUP BY month
ORDER BY month
LIMIT 10;
"""
result2=pd.read_sql(query2,conn)
print("Query 2 -Average NAV per month :")
print(result2)

Query 2 -Average NAV per month :
     month  avg_nav
0  2022-01   207.05
1  2022-02   207.72
2  2022-03   209.69
3  2022-04   211.79
4  2022-05   212.73
5  2022-06   213.86
6  2022-07   213.95
7  2022-08   215.70
8  2022-09   218.47
9  2022-10   219.53


In [32]:
query_check="""
SELECT date FROM fact_nav LIMIT 5;
"""
result_check=pd.read_sql(query_check,conn)
print(result_check)

         date
0  2022-01-03
1  2022-01-04
2  2022-01-05
3  2022-01-06
4  2022-01-07


In [34]:
sip=pd.read_csv(os.path.join(RAW_DIR,'04_monthly_sip_inflows.csv'))
sip.to_sql('fact_sip_industry',engine,if_exists='replace',index=False)
print("fact_sip_industry loaded :",len(sip),"rows")

fact_sip_industry loaded : 48 rows


SIP inflows YoY growth

In [36]:
query3="""
SELECT month,
       sip_inflow_crore,
       ROUND(yoy_growth_pct,2) as yoy_growth_pct
FROM fact_sip_industry
WHERE yoy_growth_pct IS NOT NULL
ORDER BY month
LIMIT 10;
"""
result3=pd.read_sql(query3,conn)
print("Query 3 -SIP YoY Growth :")
print(result3)

Query 3 -SIP YoY Growth :
     month  sip_inflow_crore  yoy_growth_pct
0  2023-01             13856           20.31
1  2023-02             13687           19.66
2  2023-03             14276           15.80
3  2023-04             14749           24.33
4  2023-05             14749           20.05
5  2023-06             14734           20.02
6  2023-07             15245           25.58
7  2023-08             15814           24.58
8  2023-09             16042           23.63
9  2023-10             16928           29.82


Query 4 Transactions By State

In [38]:
query4="""
SELECT state,
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount_inr)/1000000.0,2) as total_amount_millions
FROM fact_transactions
GROUP BY state
ORDER BY total_transactions DESC;
"""
result4=pd.read_sql(query4,conn)
print("Query 4-Transactions by state:")
print(result4)

Query 4-Transactions by state:
             state  total_transactions  total_amount_millions
0           Punjab                2965                 315.78
1   Madhya Pradesh                2931                 308.31
2       Tamil Nadu                2806                 315.18
3          Gujarat                2780                 298.36
4      West Bengal                2748                 297.18
5          Haryana                2736                 279.63
6        Telangana                2718                 290.22
7    Uttar Pradesh                2695                 285.37
8            Delhi                2677                 289.63
9        Karnataka                2621                 273.75
10       Rajasthan                2577                 298.65
11     Maharashtra                2524                 269.51


QUERY 5-Funds with expense ratio less than 1%


In [40]:
query5="""
SELECT scheme_name,fund_house,expense_ratio_pct,plan
FROM dim_fund
WHERE expense_ratio_pct<1.0
ORDER BY expense_ratio_pct ASC;
"""
result5=pd.read_sql(query5,conn)
print("Query 5-Funds with expense ratio<1% :")
print(result5)

Query 5-Funds with expense ratio<1% :
                                          scheme_name  \
0   Nippon India Gilt Securities Fund - Regular - ...   
1        HDFC Short Term Debt Fund - Regular - Growth   
2                Kotak Liquid Fund - Regular - Growth   
3            SBI Bluechip Fund - Direct Plan - Growth   
4           SBI Small Cap Fund - Direct Plan - Growth   
5       Nippon India Large Cap Fund - Direct - Growth   
6            ICICI Pru Liquid Fund - Regular - Growth   
7                Axis Bluechip Fund - Direct - Growth   
8        SBI Magnum Gilt Fund - Regular Plan - Growth   
9   HDFC Mid-Cap Opportunities Fund - Direct - Growth   
10                ABSL Liquid Fund - Regular - Growth   
11          ICICI Pru Bluechip Fund - Direct - Growth   
12                     Nippon India ETF Nifty 50 BeES   
13           HDFC Top 100 Fund - Direct Plan - Growth   

                  fund_house  expense_ratio_pct     plan  
0            Nippon India MF               0.55

Query 6 Top 5 funds by sharpe ratio

In [41]:
query6= """
SELECT f.scheme_name,f.fund_house,p.sharpe_ratio,p.return_3yr_pct
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code= f.amfi_code
ORDER BY p.sharpe_ratio DESC
LIMIT 5;
"""
result6=pd.read_sql(query6,conn)
print("Query 6 -Top 5 funds by Sharpe ratio:")
print(result6)


Query 6 -Top 5 funds by Sharpe ratio:
                                    scheme_name                fund_house  \
0      ICICI Pru Liquid Fund - Regular - Growth       ICICI Prudential MF   
1          Kotak Liquid Fund - Regular - Growth         Kotak Mahindra MF   
2           ABSL Liquid Fund - Regular - Growth  Aditya Birla Sun Life MF   
3  HDFC Short Term Debt Fund - Regular - Growth          HDFC Mutual Fund   
4  SBI Magnum Gilt Fund - Regular Plan - Growth           SBI Mutual Fund   

   sharpe_ratio  return_3yr_pct  
0          7.68            7.68  
1          6.18            6.18  
2          5.14            5.14  
3          1.84            7.37  
4          1.52            6.07  


Query 7 -Average SIP amount by age groups

In [42]:
query7="""
SELECT age_group,
      COUNT(*) as total_transactions,
      ROUND(AVG(amount_inr),2)as avg_sip_amount
FROM fact_transactions
WHERE transaction_type='SIP'
GROUP BY age_group
ORDER BY avg_sip_amount DESC;
"""
result7= pd.read_sql(query7,conn)
print("Query 7 -Average SIP amount by age group:")
print(result7)

Query 7 -Average SIP amount by age group:
  age_group  total_transactions  avg_sip_amount
0       56+                1478        11574.92
1     46-55                2300        11136.76
2     26-35                8063        10986.90
3     18-25                2949        10953.07
4     36-45                4926        10885.76


QUERY8 -Top 5 funds by 3 year returns

In [44]:
query8 ="""
SELECT f.scheme_name,f.fund_house,f.sub_category,
       p.return_3yr_pct
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.return_3yr_pct DESC
LIMIT 5;"""
result8=pd.read_sql(query8,conn)
print("Query 8-Top 5 funds by 3 year returns:")
print(result8)

Query 8-Top 5 funds by 3 year returns:
                                      scheme_name                fund_house  \
0      SBI Small Cap Fund - Regular Plan - Growth           SBI Mutual Fund   
1       SBI Small Cap Fund - Direct Plan - Growth           SBI Mutual Fund   
2          ABSL Small Cap Fund - Regular - Growth  Aditya Birla Sun Life MF   
3          Axis Small Cap Fund - Regular - Growth          Axis Mutual Fund   
4  Nippon India Small Cap Fund - Regular - Growth           Nippon India MF   

  sub_category  return_3yr_pct  
0    Small Cap           23.39  
1    Small Cap           23.14  
2    Small Cap           22.38  
3    Small Cap           20.98  
4    Small Cap           20.15  


QUERY 9 AUM growth by fund house over years

In [45]:
query9 = """
SELECT fund_house,
       substr(date, 1, 4) as year,
       ROUND(AVG(aum_lakh_crore), 2) as avg_aum
FROM fact_aum
GROUP BY fund_house, year
ORDER BY fund_house, year;
"""

result9 = pd.read_sql(query9, conn)
print("Query 9 — AUM growth by fund house:")
print(result9)

Query 9 — AUM growth by fund house:
                  fund_house  year  avg_aum
0   Aditya Birla Sun Life MF  2022     2.81
1   Aditya Birla Sun Life MF  2023     2.92
2   Aditya Birla Sun Life MF  2024     3.62
3   Aditya Birla Sun Life MF  2025     4.22
4           Axis Mutual Fund  2022     2.45
5           Axis Mutual Fund  2023     2.50
6           Axis Mutual Fund  2024     2.90
7           Axis Mutual Fund  2025     3.30
8            DSP Mutual Fund  2022     1.11
9            DSP Mutual Fund  2023     1.23
10           DSP Mutual Fund  2024     1.72
11           DSP Mutual Fund  2025     2.13
12          HDFC Mutual Fund  2022     4.40
13          HDFC Mutual Fund  2023     4.92
14          HDFC Mutual Fund  2024     7.14
15          HDFC Mutual Fund  2025     8.63
16       ICICI Prudential MF  2022     4.77
17       ICICI Prudential MF  2023     5.45
18       ICICI Prudential MF  2024     7.65
19       ICICI Prudential MF  2025     9.77
20         Kotak Mahindra MF  2022     2

Query 10 — Transaction split by type:

In [46]:
query10 = """
SELECT transaction_type,
       COUNT(*) as total_transactions,
       ROUND(SUM(amount_inr)/1000000.0, 2) as total_amount_millions,
       ROUND(AVG(amount_inr), 2) as avg_amount
FROM fact_transactions
GROUP BY transaction_type
ORDER BY total_transactions DESC;
"""

result10 = pd.read_sql(query10, conn)
print("Query 10 — Transaction split by type:")
print(result10)

Query 10 — Transaction split by type:
  transaction_type  total_transactions  total_amount_millions  avg_amount
0              SIP               19716                 217.23    11018.13
1          Lumpsum                8095                2059.82   254456.02
2       Redemption                4967                1244.53   250558.79


In [47]:
queries_sql = """
-- Query 1: Top 5 funds by AUM
SELECT scheme_name, fund_house, aum_crore
FROM fact_performance
ORDER BY aum_crore DESC
LIMIT 5;

-- Query 2: Average NAV per month
SELECT substr(date, 1, 7) as month,
       ROUND(AVG(nav), 2) as avg_nav
FROM fact_nav
GROUP BY month
ORDER BY month;

-- Query 3: SIP YoY growth
SELECT month, sip_inflow_crore,
       ROUND(yoy_growth_pct, 2) as yoy_growth_pct
FROM fact_sip_industry
WHERE yoy_growth_pct IS NOT NULL
ORDER BY month;

-- Query 4: Transactions by state
SELECT state, COUNT(*) as total_transactions,
       ROUND(SUM(amount_inr)/1000000.0, 2) as total_amount_millions
FROM fact_transactions
GROUP BY state
ORDER BY total_transactions DESC;

-- Query 5: Funds with expense ratio < 1%
SELECT scheme_name, fund_house, expense_ratio_pct, plan
FROM dim_fund
WHERE expense_ratio_pct < 1.0
ORDER BY expense_ratio_pct ASC;

-- Query 6: Top 5 funds by Sharpe ratio
SELECT f.scheme_name, f.fund_house, p.sharpe_ratio, p.return_3yr_pct
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.sharpe_ratio DESC
LIMIT 5;

-- Query 7: Average SIP amount by age group
SELECT age_group, COUNT(*) as total_transactions,
       ROUND(AVG(amount_inr), 2) as avg_sip_amount
FROM fact_transactions
WHERE transaction_type = 'SIP'
GROUP BY age_group
ORDER BY avg_sip_amount DESC;

-- Query 8: Top 5 funds by 3 year returns
SELECT f.scheme_name, f.fund_house, f.sub_category, p.return_3yr_pct
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.return_3yr_pct DESC
LIMIT 5;

-- Query 9: AUM growth by fund house over years
SELECT fund_house, substr(date, 1, 4) as year,
       ROUND(AVG(aum_lakh_crore), 2) as avg_aum
FROM fact_aum
GROUP BY fund_house, year
ORDER BY fund_house, year;

-- Query 10: Transaction split by type
SELECT transaction_type, COUNT(*) as total_transactions,
       ROUND(SUM(amount_inr)/1000000.0, 2) as total_amount_millions,
       ROUND(AVG(amount_inr), 2) as avg_amount
FROM fact_transactions
GROUP BY transaction_type
ORDER BY total_transactions DESC;
"""

sql_path = os.path.join(BASE_DIR, '..', 'sql', 'queries.sql')
with open(sql_path, 'w') as f:
    f.write(queries_sql)
print("queries.sql saved successfully")

queries.sql saved successfully


## Task 6 — 10 SQL Queries
1. Top 5 funds by AUM — Mirae Asset Emerging Bluechip leads at Rs.49,046 crore
2. Average NAV per month — steady growth from 207 in Jan 2022
3. SIP YoY growth — 20-30% growth in 2023
4. Transactions by state — Punjab highest, Maharashtra lowest
5. Funds with expense ratio < 1% — 14 funds, cheapest is Nippon Gilt at 0.55%
6. Top 5 by Sharpe ratio — all Debt funds due to low volatility
7. Average SIP by age group — 56+ invests highest at Rs.11,574
8. Top 5 by 3yr returns — all Small Cap funds, SBI leads at 23.39%
9. AUM growth by fund house — all doubled from 2022 to 2025, SBI largest at 12.50 lakh crore
10. Transaction split — SIP most frequent, Lumpsum highest average amount

In [49]:
data_dict = """# Data Dictionary — Bluestock MF Capstone

## 01_fund_master.csv
| Column | Type | Description |
|--------|------|-------------|
| amfi_code | TEXT | Unique AMFI scheme code |
| fund_house | TEXT | AMC name |
| scheme_name | TEXT | Full official scheme name |
| category | TEXT | Equity / Debt |
| sub_category | TEXT | Large Cap / Mid Cap / Small Cap etc. |
| plan | TEXT | Regular or Direct |
| launch_date | DATE | Fund launch date |
| benchmark | TEXT | Official benchmark index |
| expense_ratio_pct | REAL | Annual expense ratio % |
| exit_load_pct | REAL | Exit load % |
| min_sip_amount | INT | Minimum SIP amount in Rs. |
| min_lumpsum_amount | INT | Minimum lumpsum amount in Rs. |
| fund_manager | TEXT | Primary fund manager name |
| risk_category | TEXT | Low/Moderate/High/Very High |
| sebi_category_code | TEXT | SEBI internal category code |

## 02_nav_history.csv
| Column | Type | Description |
|--------|------|-------------|
| amfi_code | TEXT | Foreign key to fund_master |
| date | DATE | NAV date (business days only) |
| nav | REAL | NAV in Rs. |

## 03_aum_by_fund_house.csv
| Column | Type | Description |
|--------|------|-------------|
| date | DATE | Quarter end date |
| fund_house | TEXT | AMC name |
| aum_lakh_crore | REAL | AUM in Rs. lakh crore |
| aum_crore | INT | AUM in Rs. crore |
| num_schemes | INT | Number of schemes |

## 04_monthly_sip_inflows.csv
| Column | Type | Description |
|--------|------|-------------|
| month | TEXT | YYYY-MM format |
| sip_inflow_crore | INT | Total SIP inflows in Rs. crore |
| active_sip_accounts_crore | REAL | Active SIP accounts in crore |
| new_sip_accounts_lakh | REAL | New SIP registrations in lakh |
| sip_aum_lakh_crore | REAL | SIP AUM in Rs. lakh crore |
| yoy_growth_pct | REAL | YoY growth % in SIP inflows |

## 07_scheme_performance.csv
| Column | Type | Description |
|--------|------|-------------|
| amfi_code | TEXT | Foreign key to fund_master |
| return_1yr_pct | REAL | 1 year absolute return % |
| return_3yr_pct | REAL | 3 year CAGR % |
| return_5yr_pct | REAL | 5 year CAGR % |
| sharpe_ratio | REAL | Risk adjusted return |
| alpha | REAL | Excess return over benchmark |
| beta | REAL | Market sensitivity |
| max_drawdown_pct | REAL | Worst peak to trough decline |
| expense_ratio_pct | REAL | Annual expense ratio % |

## 08_investor_transactions.csv
| Column | Type | Description |
|--------|------|-------------|
| investor_id | TEXT | Unique investor ID |
| transaction_date | DATE | Date of transaction |
| amfi_code | TEXT | Fund invested in |
| transaction_type | TEXT | SIP / Lumpsum / Redemption |
| amount_inr | INT | Transaction amount in Rs. |
| state | TEXT | Investor state |
| city_tier | TEXT | T30 or B30 city |
| age_group | TEXT | 18-25 / 26-35 / 36-45 / 46-55 / 56+ |
| gender | TEXT | Male / Female |
| annual_income_lakh | REAL | Annual income in Rs. lakh |
"""

dict_path = os.path.join(BASE_DIR, '..', 'reports', 'data_dictionary.md')
with open(dict_path, 'w') as f:
    f.write(data_dict)
print("data_dictionary.md saved successfully")

data_dictionary.md saved successfully
